In [1]:
import json
import sys
sys.path.insert(0, "..")

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import config
from oro import esquema
from src.db_io import escribir_tabla_sqlite, leer_tabla_sqlite
from src.features_modelo import features_modelo_a, features_modelo_b
from src.fuga import validar_sin_fuga
from src.niveles import asignar_niveles_por_poblacion

df = leer_tabla_sqlite(config.ORO_DB, "cliente_features")
(config.OUTPUTS_DIR / "models").mkdir(parents=True, exist_ok=True)

CATEGORICAS = ["desc_segmento", "grupo_edad", "desc_tipo_de_vivienda"]


def preparar_matriz(datos, feature_cols, contexto):
    validar_sin_fuga(feature_cols, contexto=contexto)
    X = pd.get_dummies(
        datos[feature_cols],
        columns=[c for c in CATEGORICAS if c in feature_cols],
        dummy_na=False,
    )
    # SPEC_V2 §1.3: el guard corre también sobre las columnas REALES que llegan
    # al fit (post get_dummies), no solo sobre la lista de selección previa.
    validar_sin_fuga(X.columns, contexto=f"{contexto} (post get_dummies)")
    return X


# SPEC_V2 §6.1: AMBOS modelos se entrenan sobre toda la base apta. Los clientes
# sin productos son ejemplos negativos legítimos y necesarios, no se excluyen.
entrenables = df[df["apto_entrenamiento"] == 1].reset_index(drop=True)
y = entrenables["etiqueta_adopcion"]

cols_a = features_modelo_a(entrenables.columns)   # lanza si hay fuga
cols_b = features_modelo_b(entrenables.columns)   # lanza si hay fuga
Xa = preparar_matriz(entrenables, cols_a, "Modelo A")
Xb = preparar_matriz(entrenables, cols_b, "Modelo B")

idx_train, idx_test = train_test_split(
    np.arange(len(entrenables)), test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE, stratify=y,
)
print(f"entrenables: {len(entrenables):,} de {len(df):,}")
print(f"Modelo A: {Xa.shape[1]} features | Modelo B: {Xb.shape[1]} features")
print(f"tasa de adopción: {y.mean():.4f}")


entrenables: 860,153 de 860,223
Modelo A: 73 features | Modelo B: 30 features
tasa de adopción: 0.0717


In [2]:
def entrenar_y_evaluar(X, nombre):
    X_tr, X_te = X.iloc[idx_train], X.iloc[idx_test]
    y_tr, y_te = y.iloc[idx_train], y.iloc[idx_test]
    modelo = HistGradientBoostingClassifier(random_state=config.RANDOM_STATE)
    modelo.fit(X_tr, y_tr)
    proba_te = modelo.predict_proba(X_te)[:, 1]
    auc = float(roc_auc_score(y_te, proba_te))
    print(f"{nombre}: AUC = {auc:.4f}  (n_train={len(X_tr):,}, n_test={len(X_te):,})")
    # SPEC_V2 §1: por encima de 0.95 se asume fuga residual y se detiene el trabajo.
    # Este assert NO se debilita: si dispara, hay que investigar fuga, no subir el umbral.
    assert auc <= config.UMBRAL_AUC_FUGA, (
        f"{nombre}: AUC={auc:.4f} > {config.UMBRAL_AUC_FUGA}. "
        "SPEC_V2 §1: asumir fuga residual e investigar antes de continuar."
    )
    return modelo, auc, proba_te


modelo_a, auc_a, proba_a_te = entrenar_y_evaluar(Xa, "Modelo A (completo)")
modelo_b, auc_b, proba_b_te = entrenar_y_evaluar(Xb, "Modelo B (cold-start)")

# SPEC_V2 §6.1: también el AUC de A restringido al subconjunto CON productos, para
# separar cuánto del poder de ranking del Modelo A viene realmente de comportamiento
# transaccional frente a lo que viene de capacidad financiera compartida con B.
tiene_prod_te = entrenables.iloc[idx_test]["tiene_historial_producto"].to_numpy() == 1
y_te = y.iloc[idx_test].to_numpy()
auc_a_con_productos = float(roc_auc_score(y_te[tiene_prod_te], proba_a_te[tiene_prod_te]))
print(f"Modelo A restringido a clientes CON productos: AUC = {auc_a_con_productos:.4f} "
      f"(n={int(tiene_prod_te.sum()):,})")

metricas = {
    "modelo_a": {"auc": auc_a, "n_features": int(Xa.shape[1])},
    "modelo_b": {"auc": auc_b, "n_features": int(Xb.shape[1])},
    "modelo_a_solo_con_productos": {"auc": auc_a_con_productos,
                                    "n": int(tiene_prod_te.sum())},
    "n_entrenables": int(len(entrenables)),
    "tasa_adopcion": float(y.mean()),
}
with open(config.OUTPUTS_DIR / "models" / "metricas_propension.json", "w") as f:
    json.dump(metricas, f, indent=2)
joblib.dump(modelo_a, config.OUTPUTS_DIR / "models" / "propension_modelo_a.pkl")
joblib.dump(modelo_b, config.OUTPUTS_DIR / "models" / "propension_modelo_b.pkl")


Modelo A (completo): AUC = 0.8942  (n_train=688,122, n_test=172,031)


Modelo B (cold-start): AUC = 0.8375  (n_train=688,122, n_test=172,031)


Modelo A restringido a clientes CON productos: AUC = 0.8241 (n=105,997)


['C:\\Users\\natam\\OneDrive\\Desktop\\Prueba-Tecnica-CREAN\\outputs\\models\\propension_modelo_b.pkl']

In [3]:
# SPEC_V2 §4.6: permutation importance post-entrenamiento, más confiable que la
# importancia nativa de árboles (que sobrevalora las variables de alta cardinalidad).
frames = []
for modelo, X, nombre in [(modelo_a, Xa, "A"), (modelo_b, Xb, "B")]:
    X_te = X.iloc[idx_test]
    sub = X_te.sample(n=min(30_000, len(X_te)), random_state=config.RANDOM_STATE)
    pi = permutation_importance(
        modelo, sub, y.iloc[idx_test].loc[sub.index], n_repeats=5,
        random_state=config.RANDOM_STATE, scoring="roc_auc",
    )
    frames.append(pd.DataFrame({
        "variable": sub.columns,
        "importancia": pi.importances_mean,
        "importancia_std": pi.importances_std,
        "modelo": nombre,
    }))

importancia = pd.concat(frames, ignore_index=True).sort_values(
    ["modelo", "importancia"], ascending=[True, False])
importancia.to_csv(config.OUTPUTS_DIR / "models" / "importancia_permutacion.csv", index=False)
print(importancia.groupby("modelo").head(10).to_string(index=False))


                      variable  importancia  importancia_std modelo
       n_productos_no_etiqueta     0.016287         0.001038      A
                 total_activos     0.011697         0.000744      A
           saldo_liquido_total     0.010680         0.000760      A
   cuenta_ahorro_saldo_prom_6m     0.005137         0.000642      A
        dias_desde_ultimo_dato     0.004118         0.000892      A
            bolsillos_tenencia     0.003357         0.000355      A
               pct_dif_ingreso     0.002526         0.000631      A
            ingresos_mensuales     0.002369         0.000456      A
      fiducuenta_saldo_prom_6m     0.002140         0.000458      A
             estimador_ingreso     0.001748         0.000175      A
                 total_activos     0.044315         0.002583      B
            ingresos_mensuales     0.042842         0.002711      B
             estimador_ingreso     0.022641         0.001744      B
              total_patrimonio     0.008937     

In [4]:
# SPEC_V2 §2: ningún cliente queda sin score.
# §6.1: Modelo A se aplica a clientes con al menos un producto; Modelo B a los
# clientes sin ningún producto (incluye los 70 clientes "sin_ninguna_senal" que
# quedaron fuera de `entrenables`: también ellos deben quedar scoreados).
Xa_full = preparar_matriz(df, cols_a, "Modelo A scoring").reindex(
    columns=Xa.columns, fill_value=False)
Xb_full = preparar_matriz(df, cols_b, "Modelo B scoring").reindex(
    columns=Xb.columns, fill_value=False)

con_producto = df["tiene_historial_producto"] == 1

scores = pd.Series(np.nan, index=df.index, dtype=float)
scores[con_producto] = modelo_a.predict_proba(Xa_full[con_producto])[:, 1]
scores[~con_producto] = modelo_b.predict_proba(Xb_full[~con_producto])[:, 1]

fact = pd.DataFrame({
    "numero_id": df["numero_id"],
    "score": scores,
    "modelo_usado": np.where(con_producto, "A", "B"),
    "tiene_historial_producto": df["tiene_historial_producto"],
    "apto_entrenamiento": df["apto_entrenamiento"],
    "poblacion": np.where(con_producto, "con_historial", "sin_historial"),
})
assert fact["score"].notna().all(), "hay clientes sin score: viola SPEC_V2 §2"
assert len(fact) == len(df), "hay clientes sin fila en fact_cliente_score: viola SPEC_V2 §2"
print(f"scoreados: {len(fact):,} clientes "
      f"({int(con_producto.sum()):,} con modelo A, {int((~con_producto).sum()):,} con modelo B)")


scoreados: 860,223 clientes (529,470 con modelo A, 330,753 con modelo B)


In [5]:
# SPEC_V2 §6.2:
#  - población CON historial de inversión: ordenar por valor_esperado = score × monto_estimado_12m
#  - población SIN historial: ordenar por score, con capacidad_ahorro_anualizada como
#    referencia de valor, etiquetada explícitamente como PROXY, no como pronóstico.
# El monto (notebook 06) aún no existe: en este paso el valor de referencia
# de la población con historial es el score. El notebook 06 recalcula los niveles de
# esa población con el valor esperado y sobrescribe fact_cliente_score.
fact["capacidad_ahorro_anualizada"] = (df["capacidad_ahorro"] * 12).to_numpy()

# OBSERVACIÓN DE CALIDAD DE DATOS (solo se mide y se deja constancia; NO se limpia
# ni se winsoriza aquí — corregirlo es una decisión de negocio fuera de esta tarea,
# y reescribir el input en silencio sería peor que simplemente reportarlo).
# `capacidad_ahorro` alcanza magnitudes de billones de pesos, positivas y negativas,
# lo que no es plausible como capacidad de ahorro mensual real de una persona natural.
# Esta escala es la causa directa de lo que se mide más abajo: el proxy
# capacidad_ahorro_anualizada × score vive en el orden de 1e4-1e13, muy por encima
# del rango [0,1] del score puro que usan los ~191 clientes sin ese dato.
print("OBSERVACIÓN DE CALIDAD DE DATOS — rango de capacidad_ahorro (fuente: cliente_features):")
print(df["capacidad_ahorro"].describe().to_string())
print("\npercentiles:")
print(df["capacidad_ahorro"].quantile([0, 0.01, 0.25, 0.5, 0.75, 0.99, 1.0]).to_string())
print(
    "\nADVERTENCIA: valores del orden de ±1e12-1e13 no son plausibles como capacidad de "
    "ahorro mensual real de una persona natural. Se reporta para el registro de calidad "
    "de datos; no se corrige en esta tarea."
)

# ~191 clientes sin_historial no tienen ningún dato financiero (capacidad_ahorro
# NaN, sin_dato_financiero=1): el proxy score × capacidad_ahorro_anualizada no se
# puede calcular para ellos. En vez de dejarlos sin nivel (violaría §2), se cae de
# vuelta al score puro para ESE subconjunto, documentado con su propio tipo de
# referencia para que quede trazable que no llevan el proxy de capacidad de ahorro.
proxy_disponible = fact["capacidad_ahorro_anualizada"].notna()
fact["valor_referencia"] = np.where(
    con_producto,
    fact["score"],
    np.where(proxy_disponible, fact["capacidad_ahorro_anualizada"] * fact["score"], fact["score"]),
)
fact["tipo_valor_referencia"] = np.where(
    con_producto,
    "score_propension",
    np.where(
        proxy_disponible,
        "proxy_capacidad_ahorro_anualizada_x_score_similitud",
        "proxy_no_disponible_fallback_score_similitud",
    ),
)

fact["nivel"] = asignar_niveles_por_poblacion(fact, "valor_referencia", "poblacion")
assert fact["valor_referencia"].notna().all(), "valor_referencia con NaN: rompería la asignación de nivel"
assert fact["nivel"].notna().all(), "hay clientes sin nivel asignado: viola SPEC_V2 §2"

escribir_tabla_sqlite(fact, config.ORO_DB, "fact_cliente_score",
                      ddl=esquema.ddl_de(fact, "fact_cliente_score"),
                      indices=esquema.INDICES["fact_cliente_score"])

n_fallback = int((fact["tipo_valor_referencia"] == "proxy_no_disponible_fallback_score_similitud").sum())
print(pd.crosstab(fact["poblacion"], fact["nivel"]).to_string())
print(f"\nclientes sin_historial sin proxy de capacidad de ahorro (fallback a score puro): {n_fallback:,}")

# DIVULGACIÓN EXPLÍCITA DEL FALLBACK: el score puro de estos ~191 clientes NO
# rankea de forma comparable contra el resto de sin_historial (que rankea en
# capacidad_ahorro_anualizada × score, escalado por la magnitud medida arriba). Se
# mide y se imprime la distribución de nivel de este subgrupo en vez de dejar que
# quien consuma `nivel` aguas abajo lo descubra por su cuenta.
mascara_fallback = fact["tipo_valor_referencia"] == "proxy_no_disponible_fallback_score_similitud"
dist_nivel_fallback = fact.loc[mascara_fallback, "nivel"].value_counts()
proxy_valido_sin_hist = fact.loc[
    (fact["poblacion"] == "sin_historial")
    & (fact["tipo_valor_referencia"] == "proxy_capacidad_ahorro_anualizada_x_score_similitud"),
    "valor_referencia",
]
print(f"\nDistribución de nivel dentro del subgrupo fallback (n={n_fallback:,}):")
print(dist_nivel_fallback.to_string())
print(
    f"\nrango de valor_referencia del fallback (score puro): "
    f"[{fact.loc[mascara_fallback, 'valor_referencia'].min():.6f}, "
    f"{fact.loc[mascara_fallback, 'valor_referencia'].max():.6f}]"
)
print(
    f"percentil 25 de valor_referencia del resto de sin_historial (con proxy válido): "
    f"{proxy_valido_sin_hist.quantile(0.25):,.2f}"
)
print(
    "\nADVERTENCIA — nivel D es el ÚNICO nivel alcanzable para el subgrupo fallback: su "
    "valor_referencia (score puro, rango ~[0,1]) queda varios órdenes de magnitud por "
    "debajo del percentil 25 del resto de la población sin_historial. Por construcción "
    "no pueden competir por B, C ni mucho menos A: 'D' para este subgrupo NO es una "
    "señal de mérito relativo, es una consecuencia mecánica de mezclar una probabilidad "
    "acotada [0,1] con un producto no acotado (capacidad_ahorro_anualizada × score) en "
    "la misma población. Es una asignación defendible -no hay dato financiero para "
    "estimar su valor- pero el negocio debe leerla como 'sin dato', no como 'bajo score "
    "comparado con sus pares'."
)
print(
    "\nSPEC_V2 §6.1 — RESTRICCIÓN DE INTERPRETACIÓN: para el segmento sin productos "
    "la etiqueta es 0 por construcción (no puede haber positivos: nadie sin producto "
    "puede tener saldo activo en Invesbot/Inversión Virtual). El score del Modelo B "
    "sobre ese segmento NO fue validado contra positivos reales y por lo tanto es un "
    "PUNTAJE DE SIMILITUD (lookalike) con clientes que sí adoptaron, no una "
    "probabilidad. Se reporta y se debe consumir aguas abajo SIEMPRE como ranking "
    "relativo por niveles A/B/C/D dentro de la población sin_historial, NUNCA como "
    "porcentaje de probabilidad de adopción.\n"
    "\nSPEC_V2 §6.2 — CRITERIO DE CORTE: cuartiles del rango percentil calculados "
    "DENTRO de cada población por separado. Un cliente 'A' sin historial y un "
    "cliente 'A' con historial no son comparables entre sí: cada uno es del 25% "
    "superior de SU población, con una unidad de referencia distinta "
    "(probabilidad validada vs. proxy de similitud)."
)


OBSERVACIÓN DE CALIDAD DE DATOS — rango de capacidad_ahorro (fuente: cliente_features):
count    8.599740e+05
mean    -1.084789e+08
std      3.145494e+10
min     -8.499998e+12
25%      8.726742e+05
50%      1.517512e+06
75%      3.696837e+06
max      9.000008e+12

percentiles:
0.00   -8.499998e+12
0.01   -2.000000e+06
0.25    8.726742e+05
0.50    1.517512e+06
0.75    3.696837e+06
0.99    1.950276e+07
1.00    9.000008e+12

ADVERTENCIA: valores del orden de ±1e12-1e13 no son plausibles como capacidad de ahorro mensual real de una persona natural. Se reporta para el registro de calidad de datos; no se corrige en esta tarea.


nivel               A       B       C       D
poblacion                                    
con_historial  132368  132367  132368  132367
sin_historial   82689   82688   82688   82688

clientes sin_historial sin proxy de capacidad de ahorro (fallback a score puro): 191

Distribución de nivel dentro del subgrupo fallback (n=191):
nivel
D    191

rango de valor_referencia del fallback (score puro): [0.000735, 0.005981]
percentil 25 de valor_referencia del resto de sin_historial (con proxy válido): 33,996.87

ADVERTENCIA — nivel D es el ÚNICO nivel alcanzable para el subgrupo fallback: su valor_referencia (score puro, rango ~[0,1]) queda varios órdenes de magnitud por debajo del percentil 25 del resto de la población sin_historial. Por construcción no pueden competir por B, C ni mucho menos A: 'D' para este subgrupo NO es una señal de mérito relativo, es una consecuencia mecánica de mezclar una probabilidad acotada [0,1] con un producto no acotado (capacidad_ahorro_anualizada × score) en 

In [6]:
# Medición de masa de empates en la frontera A/B (ver src/niveles.py):
# `asignar_niveles_por_poblacion` usa rank(method="first"), que rompe empates por
# orden de aparición de la fila. Un bloque de valores de valor_referencia IDÉNTICOS
# que sea más grande que el hueco entre percentiles 25 y 75 puede quedar repartido
# entre niveles distintos por pura posición de fila, no por diferencia real de score.
# No se toca src/niveles.py: esto es medición y documentación, no un fix.
resumen_empates = []
for poblacion, grupo in fact.groupby("poblacion"):
    conteo_valores = grupo["valor_referencia"].value_counts()
    valor_modal = float(conteo_valores.idxmax())
    n_modal = int(conteo_valores.max())

    # valores de valor_referencia que aparecen en más de un nivel: la frontera
    # entre esos niveles, para ese valor, es arbitraria (depende del orden de fila).
    niveles_por_valor = grupo.groupby("valor_referencia")["nivel"].nunique()
    valores_multinivel = niveles_por_valor[niveles_por_valor > 1]

    # específicamente la frontera A/B, la de mayor relevancia de negocio (quién
    # entra a la lista de máxima prioridad y quién se queda justo fuera):
    valores_a = set(grupo.loc[grupo["nivel"] == "A", "valor_referencia"])
    valores_b = set(grupo.loc[grupo["nivel"] == "B", "valor_referencia"])
    valores_frontera_ab = valores_a & valores_b
    n_clientes_frontera_ab = int(grupo["valor_referencia"].isin(valores_frontera_ab).sum())

    resumen_empates.append({
        "poblacion": poblacion,
        "n_clientes": len(grupo),
        "valor_modal": valor_modal,
        "n_en_valor_modal": n_modal,
        "pct_en_valor_modal": n_modal / len(grupo),
        "n_valores_distintos_multinivel": int(len(valores_multinivel)),
        "n_valores_distintos_frontera_ab": len(valores_frontera_ab),
        "n_clientes_en_frontera_ab": n_clientes_frontera_ab,
        "pct_clientes_en_frontera_ab": n_clientes_frontera_ab / len(grupo),
    })

resumen_empates_df = pd.DataFrame(resumen_empates)
resumen_empates_df.to_csv(
    config.OUTPUTS_DIR / "models" / "masa_empates_frontera_niveles.csv", index=False)
print(resumen_empates_df.to_string(index=False))

for r in resumen_empates:
    if r["n_clientes_en_frontera_ab"] > 0:
        print(
            f"\nADVERTENCIA población={r['poblacion']}: {r['n_clientes_en_frontera_ab']:,} clientes "
            f"({r['pct_clientes_en_frontera_ab']:.2%} de la población) comparten valor_referencia "
            "con clientes del nivel contiguo pero caen en A vs B por orden de fila "
            "(rank method='first'), no por una diferencia real de score o valor. "
            "El corte dentro de ese bloque de empate es ARBITRARIO: el negocio debe "
            "saber que 'nivel A' en el borde no es una distinción confiable respecto "
            "a 'nivel B' en el borde para esos clientes."
        )


    poblacion  n_clientes  valor_modal  n_en_valor_modal  pct_en_valor_modal  n_valores_distintos_multinivel  n_valores_distintos_frontera_ab  n_clientes_en_frontera_ab  pct_clientes_en_frontera_ab
con_historial      529470     0.003137               420            0.000793                               0                                0                          0                          0.0
sin_historial      330753     0.000000             24825            0.075056                               0                                0                          0                          0.0


In [7]:
# Targeting de campaña: qué recall/precisión se obtiene contactando el top N%.
# Se calcula sobre el TEST del Modelo A, la única población con positivos reales
# (el segmento sin productos no tiene positivos por construcción, ver celda anterior
# sobre la restricción de interpretación del Modelo B).
orden = np.argsort(-proba_a_te)
y_ord = y.iloc[idx_test].to_numpy()[orden]
n = len(y_ord)

filas = []
for pct in [0.01, 0.05, 0.10, 0.20]:
    corte = max(1, int(np.ceil(n * pct)))
    sel = y_ord[:corte]
    filas.append({"top_pct": pct, "n_contactados": corte,
                  "precision": sel.sum() / corte,
                  "recall": sel.sum() / y_ord.sum()})
curva = pd.DataFrame(filas)
curva.to_csv(config.OUTPUTS_DIR / "models" / "curva_precision_recall.csv", index=False)
print(curva.to_string(index=False))


 top_pct  n_contactados  precision   recall
    0.01           1721   0.578152 0.080717
    0.05           8602   0.442688 0.308915
    0.10          17204   0.370205 0.516671
    0.20          34407   0.277473 0.774479


In [8]:
# D0: sensibilidad de la etiqueta de adopción a la exigencia de
# recencia. D0 mantuvo `etiqueta_adopcion` sin exigir que el saldo positivo en
# Invesbot/Inversión Virtual sea reciente, razonando que un snapshot viejo con saldo
# positivo evidencia dato desactualizado, no abandono, y que exigir recencia
# penalizaría a clientes cuyas fuentes se refrescan con menor frecuencia. D0 exige
# probar que esa elección no es determinante: reentrenar Modelo A con la etiqueta
# estricta `etiqueta_adopcion_reciente` (ventana de
# config.VENTANA_DIAS_ETIQUETA_RECIENTE días) y comparar. Solo Modelo A: en la
# población del Modelo B la etiqueta es 0 por construcción (nota SPEC_V2 §6.1
# arriba), no hay nada que comparar.
from scipy.stats import spearmanr

y_reciente = entrenables["etiqueta_adopcion_reciente"]
print(f"Positivos etiqueta principal (entrenables): {int(y.sum()):,} de {len(y):,}")
print(f"Positivos etiqueta reciente  (entrenables): {int(y_reciente.sum()):,} de {len(y_reciente):,}")

modelo_a_reciente = HistGradientBoostingClassifier(random_state=config.RANDOM_STATE)
modelo_a_reciente.fit(Xa.iloc[idx_train], y_reciente.iloc[idx_train])

proba_reciente_te = modelo_a_reciente.predict_proba(Xa.iloc[idx_test])[:, 1]
auc_reciente = float(roc_auc_score(y_reciente.iloc[idx_test], proba_reciente_te))

print(f"Modelo A (etiqueta principal):   AUC = {auc_a:.4f}")
print(f"Modelo A (etiqueta reciente, D0): AUC = {auc_reciente:.4f}")
print(f"Diferencia de AUC: {auc_reciente - auc_a:+.4f}")


Positivos etiqueta principal (entrenables): 61,636 de 860,153
Positivos etiqueta reciente  (entrenables): 45,530 de 860,153


Modelo A (etiqueta principal):   AUC = 0.8942
Modelo A (etiqueta reciente, D0): AUC = 0.8909
Diferencia de AUC: -0.0033


In [9]:
# Comparación de rankings sobre TODA la población con producto (no solo el test
# set): Spearman rho entre el score principal (ya persistido en `fact`) y el score
# bajo la etiqueta reciente, y % de clientes que cambian de nivel A/B/C/D bajo el
# mismo criterio de asignación (`asignar_niveles_por_poblacion`, cuartiles de rango
# percentil, calculados dentro de esta única población de comparación).
#
# ALINEACIÓN GARANTIZADA: ambos vectores de score se subseleccionan con la MISMA
# máscara booleana `con_producto` (definida en la celda de scoring y sin modificar
# desde entonces) aplicada a estructuras que comparten el índice original de `df`
# (`Xa_full` y `fact` se construyeron directamente sobre `df`, preservando su
# índice y orden de filas). Indexar con la misma máscara booleana selecciona las
# mismas filas, en el mismo orden relativo, en ambos casos — el reset_index(drop=True)
# solo descarta la etiqueta de índice, no reordena nada. Por construcción no puede
# haber desalineación cliente-a-cliente entre los dos vectores comparados.
scores_reciente_full = pd.Series(
    modelo_a_reciente.predict_proba(Xa_full[con_producto])[:, 1],
    index=df.index[con_producto],
)
scores_principal_full = fact.loc[con_producto, "score"].reset_index(drop=True)
scores_reciente_full = scores_reciente_full.reset_index(drop=True)

rho, p_valor = spearmanr(scores_principal_full, scores_reciente_full)

niveles_principal = asignar_niveles_por_poblacion(
    pd.DataFrame({"valor": scores_principal_full, "g": "x"}), "valor", "g")
niveles_reciente = asignar_niveles_por_poblacion(
    pd.DataFrame({"valor": scores_reciente_full, "g": "x"}), "valor", "g")
cambia_nivel = (niveles_principal.to_numpy() != niveles_reciente.to_numpy())
pct_cambia = float(cambia_nivel.mean())

sensibilidad = {
    "auc_principal": auc_a,
    "auc_reciente": auc_reciente,
    "spearman_rho": float(rho),
    "spearman_p": float(p_valor),
    "pct_clientes_cambian_nivel": pct_cambia,
    "n_clientes_comparados": int(con_producto.sum()),
}
with open(config.OUTPUTS_DIR / "models" / "sensibilidad_recencia_etiqueta.json", "w") as f:
    json.dump(sensibilidad, f, indent=2)

print(f"Spearman rho = {rho:.4f} (p = {p_valor:.2e})")
print(f"Clientes que cambian de nivel A/B/C/D: {pct_cambia:.2%} "
      f"({int(cambia_nivel.sum()):,} de {int(con_producto.sum()):,})")
print(
    "\nSPEC_V2/D0 — INTERPRETACIÓN: si el ranking se mantiene estable "
    "(rho alto, pocos cambios de nivel), la decisión de NO exigir recencia en "
    "la etiqueta no es determinante para el resultado práctico. Si cambia "
    "sustancialmente, reportar ambos escenarios al negocio en vez de uno solo."
)


Spearman rho = 0.9895 (p = 0.00e+00)
Clientes que cambian de nivel A/B/C/D: 9.71% (51,394 de 529,470)

SPEC_V2/D0 — INTERPRETACIÓN: si el ranking se mantiene estable (rho alto, pocos cambios de nivel), la decisión de NO exigir recencia en la etiqueta no es determinante para el resultado práctico. Si cambia sustancialmente, reportar ambos escenarios al negocio en vez de uno solo.


In [10]:
# D0, complemento: distancia de movimiento por nivel, no solo el % que
# cambia. Un rho alto y un 9.71% de cambio de nivel son compatibles con dos
# lecturas opuestas: (a) los clientes que cambian se mueven a la banda vecina
# junto a un corte de cuartil (jitter de discretización — D0 no es determinante),
# o (b) hay saltos distantes (top<->bottom), es decir reordenamiento real, y
# entonces SPEC_V2/D0 exige reportar AMBOS escenarios al negocio, no solo el
# preferido. La celda 5 ya documentó masa de empates pesada en valor_referencia:
# la distribución conjunta real no tiene por qué comportarse como una cópula
# gaussiana bien portada. Se mide directamente en vez de asumir una de las dos
# lecturas.
CODIGO_NIVEL = {"D": 0, "C": 1, "B": 2, "A": 3}

codigo_principal = niveles_principal.map(CODIGO_NIVEL).to_numpy()
codigo_reciente = niveles_reciente.map(CODIGO_NIVEL).to_numpy()
distancia_nivel = np.abs(codigo_principal - codigo_reciente)

conteo_distancia = pd.Series(distancia_nivel).value_counts().sort_index()
distribucion_distancia_nivel = {int(d): int(conteo_distancia.get(d, 0)) for d in range(4)}
n_distancia_2_o_mas = int((distancia_nivel >= 2).sum())
pct_distancia_2_o_mas = float((distancia_nivel >= 2).mean())

# Se agregan claves nuevas al mismo artefacto (no se restructura lo existente:
# consumidores aguas abajo pueden ya leer las claves originales).
sensibilidad["distribucion_distancia_nivel"] = distribucion_distancia_nivel
sensibilidad["n_distancia_2_o_mas"] = n_distancia_2_o_mas
sensibilidad["pct_distancia_2_o_mas"] = pct_distancia_2_o_mas
with open(config.OUTPUTS_DIR / "models" / "sensibilidad_recencia_etiqueta.json", "w") as f:
    json.dump(sensibilidad, f, indent=2)

print("Distribución de distancia de nivel (0=mismo nivel, 3=A<->D):")
for d in range(4):
    n_d = distribucion_distancia_nivel[d]
    print(f"  distancia {d}: {n_d:,} ({n_d / len(distancia_nivel):.2%})")
print(f"\nClientes con salto >= 2 niveles (p.ej. A<->C, A<->D, B<->D): "
      f"{n_distancia_2_o_mas:,} ({pct_distancia_2_o_mas:.2%})")

if n_distancia_2_o_mas == 0:
    print(
        "\nD0 — RESULTADO MEDIDO: el 100% de los cambios de nivel son adyacentes "
        "(distancia 1). No hay saltos top<->bottom. Esto es evidencia medida "
        "(no simulada) consistente con jitter de frontera de cuartil, no con "
        "reordenamiento real: la elección de D0 (no exigir recencia en la "
        "etiqueta) NO es determinante para la priorización de negocio."
    )
else:
    print(
        f"\nD0 — RESULTADO MEDIDO: {n_distancia_2_o_mas:,} clientes "
        f"({pct_distancia_2_o_mas:.2%}) saltan 2 o más niveles entre etiquetas — "
        "no es solo jitter de frontera. Esto SÍ es evidencia de reordenamiento "
        "real para ese subconjunto: SPEC_V2/D0 exige reportar AMBOS escenarios al "
        "negocio para estos clientes, no solo el preferido."
    )


Distribución de distancia de nivel (0=mismo nivel, 3=A<->D):
  distancia 0: 478,076 (90.29%)
  distancia 1: 51,045 (9.64%)
  distancia 2: 296 (0.06%)
  distancia 3: 53 (0.01%)

Clientes con salto >= 2 niveles (p.ej. A<->C, A<->D, B<->D): 349 (0.07%)

D0 — RESULTADO MEDIDO: 349 clientes (0.07%) saltan 2 o más niveles entre etiquetas — no es solo jitter de frontera. Esto SÍ es evidencia de reordenamiento real para ese subconjunto: SPEC_V2/D0 exige reportar AMBOS escenarios al negocio para estos clientes, no solo el preferido.
